## 1 (Deskewing & Slant Correction,NLP-Driven Post-Processing) iam

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForCausalLM
import timm
import Levenshtein
from tqdm import tqdm
import cv2
import numpy as np

# 1. PATHS
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_pp_1.pth"

# 2. CONFIG - Modifying heights/widths to fit Swin Window Blocks perfectly (Divisible by 32 and Window Size 7)
IMG_HEIGHT      = 112   # 112 / 32 = 3.5 -> Window adjustments processed dynamically
IMG_WIDTH       = 448   
MAX_SEQ_LEN     = 25
BEAM_SIZE       = 5
D_MODEL         = 768

def preprocess_handwriting_image(pil_img):
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(cv_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=30, maxLineGap=5)
    
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 != x1:
                angles.append(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
        if len(angles) > 0:
            angle = np.median(angles)
            
    if abs(angle) > 20: 
        angle = 0.0
        
    (h, w) = cv_img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(cv_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(cv2.cvtColor(deskewed, cv2.COLOR_GRAY2RGB))

# 3. DATASET
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, transform=None, max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words", id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}", f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                with Image.open(img_path) as open_img:
                    image = open_img.convert('RGB')
                    image.load()
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images corrupted.")

        image = preprocess_handwriting_image(image)
        if self.transform:
            image = self.transform(image)
            
        labels = self.processor(
            text, padding='max_length', max_length=self.max_target_length,
            truncation=True, return_tensors="pt"
        ).input_ids.squeeze(0)
        
        # Build mask safely using explicit token checks
        labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i] for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i] for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                with Image.open(img_path) as open_img:
                    image = open_img.convert('RGB')
                    image.load()
            except Exception:
                image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)

            image = preprocess_handwriting_image(image)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text, padding='max_length', max_length=self.max_target_length,
                truncation=True, return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
            return image, labels, text

    train_loader = DataLoader(SplitDataset(train_samples, processor, train_transform), batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(SplitDataset(val_samples, processor, val_transform), batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader


# =====================================================================
# 4. ARCHITECTURE (FIXED REAL TPS IMPLEMENTATION)
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        # Initialize to identity grid defaults
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    """ Implements actual non-rigid Thin Plate Spline mathematical warping transformations """
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.localization = LocalizationNetwork(num_control_points)
        
        # Target Grid points layout assembly configuration
        r1 = torch.linspace(-0.9, 0.9, num_control_points // 2)
        r2 = torch.tensor([-0.7, 0.7])
        grid_Y, grid_X = torch.meshgrid(r2, r1, indexing='ij')
        target_control_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1) # (N, 2)
        self.register_buffer('target_control_points', target_control_points)
        
    def _compute_partial_solutions(self, X, Y):
        N = X.size(0)
        M = Y.size(0)
        X_tiled = X.unsqueeze(1).repeat(1, M, 1)
        Y_tiled = Y.unsqueeze(0).repeat(N, 1, 1)
        dist_sq = torch.sum((X_tiled - Y_tiled) ** 2, dim=2)
        # Radial basis function configuration calculation matrix block r^2 * log(r^2)
        dist_sq[dist_sq == 0] = 1.0
        return dist_sq * torch.log(dist_sq)

    def forward(self, x):
        B, C, H, W = x.size()
        source_control_points = self.localization(x) + self.target_control_points.unsqueeze(0)
        
        # Solve dynamic TPS Matrix linear equations equations per batch item
        N = self.num_control_points
        L = torch.zeros(B, N + 3, N + 3, device=x.device)
        K = self._compute_partial_solutions(self.target_control_points, self.target_control_points)
        L[:, :N, :N] = K.unsqueeze(0)
        P = torch.cat([torch.ones(N, 1, device=x.device), self.target_control_points], dim=1)
        L[:, :N, N:] = P.unsqueeze(0)
        L[:, N:, :N] = P.t().unsqueeze(0)
        
        Y = torch.zeros(B, N + 3, 2, device=x.device)
        Y[:, :N, :] = source_control_points
        
        # Solve system mapping
        W_coef = torch.linalg.solve(L, Y)
        
        # Evaluate target coordinate grid points
        grid_Y, grid_X = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
        eval_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1).to(x.device) # (H*W, 2)
        
        K_eval = self._compute_partial_solutions(eval_points, self.target_control_points) # (H*W, N)
        P_eval = torch.cat([torch.ones(H * W, 1, device=x.device), eval_points], dim=1) # (H*W, 3)
        
        transformed_points = torch.matmul(K_eval, W_coef[:, :N, :]) + torch.matmul(P_eval, W_coef[:, N:, :])
        grid = transformed_points.view(B, H, W, 2)
        
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)  # Stage 3 Swin-Base has 512 channels
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3 feature extraction
        B, H, W, C = feat.shape            # Swin outputs [B, H, W, C] natively!
        feat       = feat.view(B, H*W, C)  # Flatten spatial dimensions safely
        return self.norm(self.proj(feat))


class RoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        roberta = RobertaModel.from_pretrained("roberta-base")
        config  = RobertaConfig.from_pretrained("roberta-base")
        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size
        self.decoder = RobertaForCausalLM(config)
        self.decoder.roberta.embeddings.load_state_dict(roberta.embeddings.state_dict())
        for i, layer in enumerate(self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(roberta.encoder.layer[i].state_dict(), strict=False)
        del roberta

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids, encoder_hidden_states=encoder_hidden_states, labels=labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)
        self.bilstm          = nn.LSTM(
            input_size=d_model, hidden_size=d_model // 2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.roberta_decoder = RoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        
        # Fixed label indexing strategy offsets safely
        dec_input = target_ids[:, :-1].clone()
        labels = target_ids[:, 1:].clone()
        
        dec_input[dec_input == -100] = 1 # Map padding targets to explicit pad token safely
        
        outputs = self.roberta_decoder(input_ids=dec_input, encoder_hidden_states=memory)
        logits = outputs.logits

        if criterion is not None:
            return criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate_greedy(self, images, max_length=MAX_SEQ_LEN, bos_token_id=0, eos_token_id=2):
        """Fully-vectorized batch greedy generation with confidence score extraction"""
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)
        
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        
        # Track cumulative probabilities and sequence lengths for averaging
        total_probs = torch.zeros(B, device=device)
        seq_lengths = torch.zeros(B, device=device)
    
        for _ in range(max_length - 1):
            outputs = self.roberta_decoder(input_ids=generated, encoder_hidden_states=memory)
            next_token_logits = outputs.logits[:, -1, :]
            
            # Convert logits to normalized probabilities
            probs = F.softmax(next_token_logits, dim=-1)
            
            # Get the highest probability token and its actual probability value
            next_token_probs, next_tokens = probs.max(dim=-1, keepdim=True)
            
            generated = torch.cat([generated, next_tokens], dim=1)
            
            # Only accumulate confidence metrics for tokens generated before EOS
            active_mask = ~finished
            total_probs += next_token_probs.squeeze(-1) * active_mask
            seq_lengths += active_mask.float()
            
            finished |= (next_tokens.squeeze(-1) == eos_token_id)
            if finished.all():
                break
    
        # Calculate average confidence per sequence (prevent division by zero)
        seq_lengths = torch.clamp(seq_lengths, min=1.0)
        confidence_scores = (total_probs / seq_lengths).cpu().numpy()
        
        return generated[:, 1:], confidence_scores

# 5. IMPROVED AGENTIC VERIFICATION MODULE
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon   = {}   
        self.freq_max  = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = {}
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] = freq.get(word, 0) + 1
        self.lexicon  = freq
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words.")

    def _normalized_freq(self, word):
        return self.lexicon.get(word, 0) / self.freq_max

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.85):
        # Clean out subword tokenizer padding artifacts and whitespace blocks
        cleaned = text_output.replace("Ġ", "").strip().lower()

        # Ignore empty strings, numbers, or short abbreviations
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return cleaned

        # Bypass correction if the deep learning system was highly confident
        if confidence is not None and confidence >= confidence_threshold:
            return cleaned

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word in self.lexicon:
            # Optimize performance: skip distant vocabulary words early
            if abs(len(word) - target_len) > 2:
                continue
            
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2: # Allow slightly looser edits for handwriting drops
                continue

            freq_score = self._normalized_freq(word)
            # Penalize long edit gaps heavily over pure language frequency
            score      = freq_score - (dist * 1.2)

            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return cleaned

        # Retain original case styling
        if text_output.strip().isupper():
            return best_candidate.upper()
        elif text_output.strip()[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 6. METRICS
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 7. LLRD OPTIMIZER
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5, decay_factor=0.75, weight_decay=0.05):
    lr0 = base_lr
    lr1 = base_lr * decay_factor
    lr2 = base_lr * (decay_factor ** 2)
    lr3 = base_lr * (decay_factor ** 3)
    lr4 = base_lr * (decay_factor ** 4)
    lr5 = base_lr * (decay_factor ** 5)
    lr6 = base_lr * (decay_factor ** 6)

    assigned = set()

    decoder_params = []
    for name, param in model.roberta_decoder.named_parameters():
        if id(param) not in assigned:
            decoder_params.append(param)
            assigned.add(id(param))

    bilstm_params = []
    for name, param in model.bilstm.named_parameters():
        if id(param) not in assigned:
            bilstm_params.append(param)
            assigned.add(id(param))

    swin_proj_params = []
    for name, param in model.swin_encoder.named_parameters():
        if id(param) not in assigned and (name.startswith("proj.") or name.startswith("norm.")):
            swin_proj_params.append(param)
            assigned.add(id(param))

    swin_s4_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_3" in name:
            swin_s4_params.append(param)
            assigned.add(id(param))

    swin_s3_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_2" in name:
            swin_s3_params.append(param)
            assigned.add(id(param))

    swin_s2_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_1" in name:
            swin_s2_params.append(param)
            assigned.add(id(param))

    swin_s1_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned:
            swin_s1_params.append(param)
            assigned.add(id(param))

    tps_params = []
    for name, param in model.tps_stn.named_parameters():
        if id(param) not in assigned:
            tps_params.append(param)
            assigned.add(id(param))

    param_groups = [
        {"params": decoder_params,   "lr": lr0, "name": "roberta_decoder"},
        {"params": bilstm_params,    "lr": lr1, "name": "bilstm"},
        {"params": swin_proj_params, "lr": lr2, "name": "swin_proj"},
        {"params": swin_s4_params,   "lr": lr3, "name": "swin_stage4"},
        {"params": swin_s3_params,   "lr": lr4, "name": "swin_stage3"},
        {"params": swin_s2_params,   "lr": lr5, "name": "swin_stage2"},
        {"params": swin_s1_params,   "lr": lr6, "name": "swin_stage1_embed"},
        {"params": tps_params,       "lr": lr2, "name": "tps_stn"},
    ]

    param_groups = [g for g in param_groups if len(g["params"]) > 0]

    print("\nLLRD Optimizer Groups:")
    print(f"{'Group':<22} {'LR':>10} {'Params':>10}")
    print("-" * 47)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<22} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    return optimizer


# =====================================================================
# 8. TRAINING
# =====================================================================
def run_training_pipeline(epochs=50, batch_size=32, base_lr=5e-5,
                         early_stopping_patience=8, resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total_p/1e6:.2f}M")
    print(f"Trainable params: {trainable_p/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    optimizer = build_llrd_optimizer(model, base_lr=base_lr, decay_factor=0.75, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max  = epochs,   # decay over full training run
    eta_min = 1e-7
)

    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    early_stopper  = EarlyStopping(patience  = 12,min_delta = 0.1)
    best_val_wer   = float('inf')
    start_epoch    = 1

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        print(f"Resumed at epoch {start_epoch} | Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")
    print(f"Input resolution: {IMG_HEIGHT}x{IMG_WIDTH}")

    for epoch in range(start_epoch, epochs + 1):
        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")

        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        lrs = [f"{g['lr']:.2e}" for g in optimizer.param_groups]
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | LRs: decoder={lrs[0]} bilstm={lrs[1]}")

        # ── Validate (Optimized via Vectorized Greedy Decoding) ──
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done = False
        
        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                
                # UNPACK BOTH matrix structures now
                tokens, confidences = model.generate_greedy(
                    images,
                    max_length=MAX_SEQ_LEN,
                    bos_token_id=tokenizer.bos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
        
                for i in range(images.size(0)):
                    raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    sanitized_raw = raw.replace("Ġ", "").strip()   
                    
                    # PASS THE ACTUAL SCORE calculated for this specific image text
                    verified = agent_verifier.verify_and_correct(
                        sanitized_raw, 
                        confidence=confidences[i],
                        confidence_threshold=0.85 # Tweak based on your preferences
                    )                   
                    all_preds.append(verified)
                    all_targets.append(text_labels[i].strip())

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            print(f"New Best WER achieved! Saving configuration parameters...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
            }, CHECKPOINT_PATH)

        if early_stopper(current_wer):
            print("Early stopping triggered. Halting run execution.")
            break

# if __name__ == "__main__":
#     run_training_pipeline(epochs=50, batch_size=32,resume_from=CHECKPOINT_PATH)

/home/mca24-26/handwritten text detection/sbenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =====================================================================
# TESTING CODE – BEAM SEARCH + AGENTIC VERIFICATION
# Copy and run after training (or in a separate cell)
# =====================================================================

import os
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer
from PIL import Image
import Levenshtein
from tqdm import tqdm
import numpy as np
from collections import defaultdict
import cv2

# =====================================================================
# CONFIG (must match training)
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_pp_1.pth"
IMG_HEIGHT = 112
IMG_WIDTH  = 448
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5
D_MODEL     = 768

# =====================================================================
# PREPROCESSING (same as training)
# =====================================================================
def preprocess_handwriting_image(pil_img):
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(cv_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=30, maxLineGap=5)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 != x1:
                angles.append(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
        if angles:
            angle = np.median(angles)
    if abs(angle) > 20:
        angle = 0.0
    (h, w) = cv_img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(cv_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(cv2.cvtColor(deskewed, cv2.COLOR_GRAY2RGB))

# =====================================================================
# TEST DATASET (reuses preprocessing)
# =====================================================================
class IAMTestDataset:
    def __init__(self, data_dir, words_txt_path, processor, transform=None,
                 max_target_length=MAX_SEQ_LEN, test_ratio=0.1):
        self.data_dir = data_dir
        self.processor = processor
        self.transform = transform
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_test_set(words_txt_path, test_ratio)

    def _parse_test_set(self, words_txt_path, test_ratio):
        if not os.path.exists(words_txt_path):
            print(f"Error: {words_txt_path} not found.")
            return
        all_samples = []
        with open(words_txt_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                word_id = parts[0]
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words", id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}", f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    all_samples.append((img_path, transcription))
        test_size = int(len(all_samples) * test_ratio)
        self.samples = all_samples[-test_size:] if test_size > 0 else all_samples
        print(f"Test samples loaded: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            with Image.open(img_path) as open_img:
                image = open_img.convert('RGB')
                image.load()
        except Exception:
            image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
        image = preprocess_handwriting_image(image)
        if self.transform:
            image = self.transform(image)
        return image, text, img_path


def get_test_dataloader(data_dir, words_txt_path, processor, batch_size=32):
    test_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_dataset = IAMTestDataset(data_dir, words_txt_path, processor, transform=test_transform, test_ratio=0.1)
    def collate_fn(batch):
        images = torch.stack([b[0] for b in batch])
        texts = [b[1] for b in batch]
        paths = [b[2] for b in batch]
        return images, texts, paths
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_fn)
    return test_loader

# =====================================================================
# BEAM SEARCH DECODER (uses model internals)
# =====================================================================
@torch.no_grad()
def beam_search_decode(model, images, tokenizer, beam_size=BEAM_SIZE, max_length=MAX_SEQ_LEN):
    """
    Custom beam search using model._extract_visual_memory and roberta_decoder.
    Returns tokens for each image in the batch.
    """
    device = images.device
    B = images.size(0)
    memory = model._extract_visual_memory(images)  # (B, seq, d_model)
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id

    all_results = []
    for b in range(B):
        mem = memory[b:b+1]                # (1, seq, d_model)
        beams = [(0.0, [bos_id])]          # (log_prob, tokens)
        completed = []

        for _ in range(max_length - 1):
            candidates = []
            for score, tokens in beams:
                if tokens[-1] == eos_id:
                    completed.append((score, tokens))
                    continue
                inp = torch.tensor([tokens], dtype=torch.long, device=device)
                out = model.roberta_decoder(
                    input_ids=inp,
                    encoder_hidden_states=mem,
                    labels=None
                )
                log_probs = F.log_softmax(out.logits[0, -1], dim=-1)  # (vocab)
                top_log_probs, top_indices = log_probs.topk(beam_size)
                for lp, idx in zip(top_log_probs.tolist(), top_indices.tolist()):
                    candidates.append((score + lp, tokens + [idx]))

            if not candidates:
                break

            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = []
            for score, tokens in candidates[:beam_size * 2]:
                if tokens[-1] == eos_id:
                    completed.append((score, tokens))
                else:
                    beams.append((score, tokens))
                if len(beams) == beam_size:
                    break
            if not beams:
                break

        all_beams = completed if completed else beams
        best_score, best_tokens = max(all_beams, key=lambda x: x[0])
        result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)  # strip BOS
        all_results.append(result)

    # Pad to same length
    max_len = max(r.size(0) for r in all_results)
    padded = torch.full((B, max_len), eos_id, dtype=torch.long, device=device)
    for i, r in enumerate(all_results):
        padded[i, :r.size(0)] = r
    return padded

# =====================================================================
# AGENTIC VERIFICATION MODULE (copy from training or import)
# =====================================================================
# (Assuming it's already defined in the notebook; if not, paste it here)
# I'll paste it for completeness – but you can skip if already defined.

class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = {}
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] = freq.get(word, 0) + 1
        self.lexicon = freq
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words.")

    def _normalized_freq(self, word):
        return self.lexicon.get(word, 0) / self.freq_max

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.85):
        cleaned = text_output.replace("Ġ", "").strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return cleaned
        if confidence is not None and confidence >= confidence_threshold:
            return cleaned

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word in self.lexicon:
            if abs(len(word) - target_len) > 2:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2:
                continue
            freq_score = self._normalized_freq(word)
            score = freq_score - (dist * 1.2)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return cleaned

        if text_output.strip().isupper():
            return best_candidate.upper()
        elif text_output.strip()[0].isupper():
            return best_candidate.capitalize()
        return best_candidate

# =====================================================================
# METRICS
# =====================================================================
def calculate_detailed_metrics(predictions, targets):
    if not targets:
        return {}
    total_chars_correct = 0
    total_chars_target = 0
    total_chars_pred = 0
    total_words_correct = 0
    total_words = len(targets)
    cer_per_sample = []
    wer_per_sample = []
    word_distances = []

    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        dist = Levenshtein.distance(pred, target)
        target_len = len(target)
        if target_len > 0:
            cer = dist / target_len
            cer_per_sample.append(cer)
            total_chars_correct += (target_len - dist)
            total_chars_target += target_len
            total_chars_pred += len(pred)
        word_distances.append(dist)
        if pred == target:
            total_words_correct += 1
            wer_per_sample.append(0.0)
        else:
            wer_per_sample.append(1.0)

    avg_cer = np.mean(cer_per_sample) * 100 if cer_per_sample else 0
    avg_wer = np.mean(wer_per_sample) * 100 if wer_per_sample else 0
    cer_standard = (total_chars_target - total_chars_correct) / total_chars_target * 100 if total_chars_target > 0 else 0

    return {
        "CER": round(avg_cer, 4),
        "CER_standard": round(cer_standard, 4),
        "WER": round(avg_wer, 4),
        "Word_Accuracy": round((total_words_correct / total_words) * 100, 4),
        "Total_Samples": total_words,
        "Correct_Words": total_words_correct,
        "Avg_Target_Length": round(total_chars_target / total_words, 2),
        "Avg_Pred_Length": round(total_chars_pred / total_words, 2),
        "Avg_Levenshtein_Distance": round(np.mean(word_distances), 2),
        "Median_Levenshtein_Distance": round(np.median(word_distances), 2)
    }

# =====================================================================
# MAIN TEST FUNCTION
# =====================================================================
def test_with_beam_search(checkpoint_path=None, beam_size=BEAM_SIZE, batch_size=32):
    print("="*60)
    print("TESTING WITH BEAM SEARCH")
    print("="*60)

    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id}")

    # DataLoader
    test_loader = get_test_dataloader(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    # Load checkpoint
    ckpt_path = checkpoint_path or CHECKPOINT_PATH
    if os.path.exists(ckpt_path):
        print(f"Loading checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"  Epoch: {ckpt.get('epoch', 'N/A')} | Best WER: {ckpt.get('best_wer', 'N/A')}%")
    else:
        print(f"Checkpoint not found: {ckpt_path}")
        return

    model.eval()
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)

    all_preds_raw = []
    all_preds_verified = []
    all_targets = []
    all_paths = []

    correction_count = 0

    for images, texts, paths in tqdm(test_loader, desc="Beam search inference"):
        images = images.to(device)
        # Beam search
        tokens = beam_search_decode(model, images, tokenizer, beam_size=beam_size)
        # Decode
        for i in range(len(tokens)):
            raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
            sanitized = raw.replace("Ġ", "").strip()
            # Agentic verification (no confidence available – set None)
            verified = agent_verifier.verify_and_correct(sanitized, confidence=None)
            if sanitized != verified:
                correction_count += 1
            all_preds_raw.append(sanitized)
            all_preds_verified.append(verified)
            all_targets.append(texts[i].strip())
            all_paths.append(paths[i])

    # Metrics
    metrics_raw = calculate_detailed_metrics(all_preds_raw, all_targets)
    metrics_verified = calculate_detailed_metrics(all_preds_verified, all_targets)

    print("\n" + "="*70)
    print(f"BEAM SEARCH RESULTS (beam_size={beam_size})")
    print("="*70)
    print(f"{'Metric':<25} {'Raw':>15} {'Verified':>15} {'Improvement':>15}")
    print("-"*70)
    cer_imp = metrics_raw['CER'] - metrics_verified['CER']
    wer_imp = metrics_raw['WER'] - metrics_verified['WER']
    acc_imp = metrics_verified['Word_Accuracy'] - metrics_raw['Word_Accuracy']
    print(f"{'CER (%)':<25} {metrics_raw['CER']:>15.2f} {metrics_verified['CER']:>15.2f} {cer_imp:>+15.2f}")
    print(f"{'WER (%)':<25} {metrics_raw['WER']:>15.2f} {metrics_verified['WER']:>15.2f} {wer_imp:>+15.2f}")
    print(f"{'Word Acc (%)':<25} {metrics_raw['Word_Accuracy']:>15.2f} {metrics_verified['Word_Accuracy']:>15.2f} {acc_imp:>+15.2f}")
    print(f"\nCorrections made by agent: {correction_count}/{len(all_preds_raw)}")

    # Save results
    out_dir = f"./test_results_beam{beam_size}"
    os.makedirs(out_dir, exist_ok=True)

    with open(os.path.join(out_dir, "metrics_raw.json"), 'w') as f:
        json.dump(metrics_raw, f, indent=2)
    with open(os.path.join(out_dir, "metrics_verified.json"), 'w') as f:
        json.dump(metrics_verified, f, indent=2)

    # Detailed predictions
    with open(os.path.join(out_dir, "predictions.txt"), 'w') as f:
        f.write(f"{'Raw':<35} {'Verified':<35} {'Target':<35}\n")
        f.write("="*105 + "\n")
        for raw, ver, tgt in zip(all_preds_raw, all_preds_verified, all_targets):
            f.write(f"{raw:<35} {ver:<35} {tgt:<35}\n")

    print(f"\nResults saved to {out_dir}")

    return metrics_raw, metrics_verified, all_preds_verified, all_targets

# =====================================================================
# RUN
# =====================================================================
if __name__ == "__main__":
    # Test with beam size 5 (you can change)
    test_with_beam_search(beam_size=5, batch_size=32)

TESTING WITH BEAM SEARCH
BOS=0 | EOS=2
Test samples loaded: 3830


/home/mca24-26/handwritten text detection/sbenv/lib/python3.8/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint: /home/mca24-26/handwritten text detection/iam_htr_pp_1.pth
  Epoch: 49 | Best WER: 25.6069%
Lexicon built: 6231 unique words.


Beam search inference: 100%|██████████████████████████| 120/120 [1:12:41<00:00, 36.35s/it]


BEAM SEARCH RESULTS (beam_size=5)
Metric                                Raw        Verified     Improvement
----------------------------------------------------------------------
CER (%)                              0.78            0.81           -0.03
WER (%)                              1.46            1.44           +0.03
Word Acc (%)                        98.54           98.56           +0.03

Corrections made by agent: 269/3830

Results saved to ./test_results_beam5
